In [1]:
import torch
import torchvision
import datasets

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Datasets:", datasets.__version__)

Torch: 2.11.0+cu128
Torchvision: 0.26.0+cu128
Datasets: 4.0.0


In [2]:
!pip uninstall -y torchvision

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128


In [3]:
!pip install -q transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.6 MB/s eta 0:00:00


## PROYECTO : detectar toxicidad en comentarios online

La detección de toxicidad permite **identificar lenguaje ofensivo, insultos o discursos de odio** en redes sociales, foros o comentarios.

Es fundamental para:

- 🛡️ **Proteger comunidades online** (YouTube, Wikipedia, X, Reddit).  
- ⚙️ **Filtrar contenido automáticamente.**  
- 🤖 **Entrenar moderadores automáticos con IA.**

## Exploración del dataset Jigsaw Toxic Comment Classification

In [4]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
dataset = load_dataset("jhan21/jigsaw-toxic-comment-classification", split="train[:2%]")
dataset = dataset.shuffle(seed=42)
dataset = dataset.train_test_split(test_size=0.2)

dataset

train.csv:   0%|          | 0.00/68.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/159571 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'],
        num_rows: 2552
    })
    test: Dataset({
        features: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'],
        num_rows: 639
    })
})

In [5]:
print(dataset)
print(dataset["train"].features)

DatasetDict({
    train: Dataset({
        features: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'],
        num_rows: 2552
    })
    test: Dataset({
        features: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'],
        num_rows: 639
    })
})
{'id': Value('string'), 'comment_text': Value('string'), 'toxic': Value('int64'), 'severe_toxic': Value('int64'), 'obscene': Value('int64'), 'threat': Value('int64'), 'insult': Value('int64'), 'identity_hate': Value('int64')}


# Configuración y Entrenamiento del Modelo BERT

## 📘 Carga del modelo preentrenado

In [7]:
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast

model_name = "distilbert-base-uncased"

tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)

model = DistilBertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 📘 Preparación de DataLoader y pipeline de entrenamiento

In [8]:
def tokenize(batch):
    return tokenizer(
        batch["comment_text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=["id", "comment_text", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
)

tokenized_dataset = tokenized_dataset.rename_column("toxic", "labels")

Map:   0%|          | 0/2552 [00:00<?, ? examples/s]

Map:   0%|          | 0/639 [00:00<?, ? examples/s]

In [9]:
print(tokenized_dataset)
print(tokenized_dataset["train"].features)

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 2552
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 639
    })
})
{'labels': Value('int64'), 'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8'))}


## Entrenamiento de Trainer con Hugging Face

In [12]:
from transformers import TrainingArguments, Trainer

new_model_name = "codigog7/toxicidad-g7"
# Configurar entrenamiento
training_args = TrainingArguments(
    output_dir=new_model_name,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"]
)


In [13]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.153279,0.160718
2,0.067396,0.190292


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=638, training_loss=0.13134608597591005, metrics={'train_runtime': 80.3505, 'train_samples_per_second': 63.522, 'train_steps_per_second': 7.94, 'total_flos': 169028400685056.0, 'train_loss': 0.13134608597591005, 'epoch': 2.0})

# guardamos el modelo

In [14]:
trainer.save_model("./modelo_toxic")
tokenizer.save_pretrained("./modelo_toxic")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./modelo_toxic/tokenizer_config.json', './modelo_toxic/tokenizer.json')

# cargamos el modelo

In [15]:
from transformers import DistilBertForSequenceClassification
from transformers import DistilBertTokenizerFast
import torch

model_path = "./modelo_toxic"

model = DistilBertForSequenceClassification.from_pretrained(model_path)
tokenizer = DistilBertTokenizerFast.from_pretrained(model_path)

model.eval()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


# probamos el modelo

In [16]:
import torch
import torch.nn.functional as F

def predecir(texto):

    inputs = tokenizer(
        texto,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=1)

    clase = torch.argmax(probs, dim=1).item()

    confianza = probs[0][clase].item()

    etiquetas = {
        0: "NO TÓXICO",
        1: "TÓXICO"
    }

    print(f"Texto: {texto}")
    print(f"Predicción: {etiquetas[clase]}")
    print(f"Confianza: {confianza:.4f}")

    return clase, confianza

In [17]:
predecir("I love your work, congratulations!")

Texto: I love your work, congratulations!
Predicción: NO TÓXICO
Confianza: 0.9976


(0, 0.9976451992988586)

In [18]:
predecir("You are stupid and nobody likes you")

Texto: You are stupid and nobody likes you
Predicción: TÓXICO
Confianza: 0.9935


(1, 0.9935246706008911)

In [19]:
trainer.save_model(new_model_name)
tokenizer.save_pretrained(new_model_name)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('codigog7/toxicidad-g7/tokenizer_config.json',
 'codigog7/toxicidad-g7/tokenizer.json')

# PUBLICAMOS EL MODELO EN HUGGING FACE

In [21]:
!hf auth login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? [y/N]: n
Token is valid (permission: write).
The token `codigog7_write` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `codigog7

In [22]:
trainer.push_to_hub(new_model_name)

print(f"¡El modelo ha sido re-publicado exitosamente en Hugging Face con el nombre '{new_model_name}'!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...idad-g7/model.safetensors:   0%|          |  575kB /  268MB            

  ...idad-g7/training_args.bin:   1%|          |  43.0B / 5.20kB            

¡El modelo ha sido re-publicado exitosamente en Hugging Face con el nombre 'codigog7/toxicidad-g7'!
